
# RetailPulse Delta Operations

Purpose:
Demonstrate Delta Lake transactional features.

Features:
- MERGE
- DESCRIBE HISTORY
- TIME TRAVEL
- DELETE
- RESTORE

In [0]:
%sql
SHOW TABLES IN workspace.retailpulse


###Create New Incoming Data to perform MERGE Operations:

In [0]:
incoming_sales_data = [
    (2, 2, 102, 2, "2026-06-01"),  # Existing sale_id=2 -> quantity changed from 1 to 2
    (6, 1, 103, 1, "2026-06-04")   # New sale_id=6
]

incoming_sales_df = spark.createDataFrame(
    incoming_sales_data,
    ["sale_id", "customer_id", "product_id", "quantity", "sale_date"]
)

display(incoming_sales_df)

In [0]:
# Save in the temp view:

incoming_sales_df.createOrReplaceTempView(
    "incoming_sales"
)

In [0]:
%sql
SELECT *
FROM workspace.retailpulse.bronze_sales
ORDER BY sale_id;

In [0]:
%sql
MERGE INTO workspace.retailpulse.bronze_sales AS target
USING incoming_sales AS source
ON target.sale_id = source.sale_id

WHEN MATCHED THEN
UPDATE SET
    target.customer_id = source.customer_id,
    target.product_id = source.product_id,
    target.quantity = source.quantity,
    target.sale_data = source.sale_date

WHEN NOT MATCHED THEN
INSERT (
    sale_id,
    customer_id,
    product_id,
    quantity,
    sale_data
)
VALUES (
    source.sale_id,
    source.customer_id,
    source.product_id,
    source.quantity,
    source.sale_date
);

In [0]:
%sql

DESCRIBE workspace.retailpulse.bronze_sales;

In [0]:
%sql
-- Magic Trick to see the old version from the DataBricks:

select * from workspace.retailpulse.bronze_sales
version as of 0
order by sale_id;



In [0]:
%sql
-- Same like Time Travelling to the next version;

select * from workspace.retailpulse.bronze_sales
version as of 1
order by sale_id;




## Let's intentionally break the table 😈
- For trying the Backup going back I mean restore the table. Once we are wrongly deleted sense.

In [0]:
%sql

DELETE FROM workspace.retailpulse.bronze_sales
where sale_id = 6;

In [0]:
%sql
-- Now we are checking

Select * from workspace.retailpulse.bronze_sales
order by sale_id;


### Then We'll Recover It

In [0]:
%sql

DESCRIBE HISTORY workspace.retailpulse.bronze_sales;

In [0]:
%sql

RESTORE TABLE workspace.retailpulse.bronze_sales
TO VERSION AS OF 2;

In [0]:
%sql

-- GOt the deleted row back

SELECT * FROM workspace.retailpulse.bronze_sales
order by sale_id;